# 04 — Spatial Weights

Construct polity-similarity spatial weight matrices and add 10 spatial lag
variables to each of the 25 intervention-coded directed-dyad files.

**Reference R scripts**: `zzz-old_version/Paper-Shadow/R/10-makeWpol.R`, `11-addSpatial.R`

**Input**: `data/interim/dd_int_{cy}_{ud}.parquet` (25 files)

**Output**: `data/interim/dd_spat_{cy}_{ud}.parquet` (25 files)

**W matrix**: For each year, W[i,j] = 1 / |polity2_i − polity2_j|  
(0 on diagonal; 0 if equal polity). Row-normalised. Computed over potential
intervener states (B side).

| Variable | Meaning |
|---|---|
| `spat_gov` | Polity-weighted fraction of B states coded gov-biased |
| `spat_opp` | Polity-weighted fraction coded opp-biased |
| `spat_US_G` | 1 if USA intervenes gov-biased |
| `spat_USSR_G` | 1 if USSR/Russia intervenes gov-biased |
| `spat_US_O` | 1 if USA intervenes opp-biased |
| `spat_USSR_O` | 1 if USSR/Russia intervenes opp-biased |
| `spat_US_USRG` | B==USA when USSR is gov-biased |
| `spat_US_USRO` | B==USA when USSR is opp-biased |
| `spat_USR_USG` | B==USSR when USA is gov-biased |
| `spat_USR_USO` | B==USSR when USA is opp-biased |

In [1]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from shadow.data.spatial import add_spatial_lags, SPAT_COLS

warnings.filterwarnings('ignore')

INTERIM  = Path('../data/interim')
N_IMP_CY = 5
N_IMP_UD = 5

print('Setup complete.')

Setup complete.


## § 1 — Main Loop: Add Spatial Lags to All 25 Files

In [2]:
t0 = time.time()

for i_cy in range(1, N_IMP_CY + 1):
    for i_ud in range(1, N_IMP_UD + 1):
        in_path  = INTERIM / f'dd_int_{i_cy}_{i_ud}.parquet'
        out_path = INTERIM / f'dd_spat_{i_cy}_{i_ud}.parquet'

        dd      = pd.read_parquet(in_path)
        dd_spat = add_spatial_lags(dd)
        dd_spat.to_parquet(out_path, index=False)

        onset  = dd_spat[dd_spat['onset_A'] == 1]
        n_spat = onset['spat_gov'].notna().sum()
        print(
            f'  dd_spat_{i_cy}_{i_ud}: {len(dd_spat):,} rows'
            f'  | onset with spat lags: {n_spat:,}'
            f'  | cols: {dd_spat.shape[1]}'
        )

elapsed = time.time() - t0
print(f'\nAll {N_IMP_CY * N_IMP_UD} dd_spat files written in {elapsed:.1f}s.')

  dd_spat_1_1: 1,198,730 rows  | onset with spat lags: 25,725  | cols: 128


  dd_spat_1_2: 1,198,730 rows  | onset with spat lags: 25,725  | cols: 128


  dd_spat_1_3: 1,198,730 rows  | onset with spat lags: 25,725  | cols: 128


  dd_spat_1_4: 1,198,730 rows  | onset with spat lags: 25,725  | cols: 128


  dd_spat_1_5: 1,198,730 rows  | onset with spat lags: 25,725  | cols: 128


...

  dd_spat_5_1: 1,198,730 rows  | onset with spat lags: 25,725  | cols: 128


  dd_spat_5_2: 1,198,730 rows  | onset with spat lags: 25,725  | cols: 128


  dd_spat_5_3: 1,198,730 rows  | onset with spat lags: 25,725  | cols: 128


  dd_spat_5_4: 1,198,730 rows  | onset with spat lags: 25,725  | cols: 128


  dd_spat_5_5: 1,198,730 rows  | onset with spat lags: 25,725  | cols: 128

All 25 dd_spat files written in 83.1s.


## § 2 — Validation

In [3]:
import glob as _glob

spat_files = sorted(_glob.glob(str(INTERIM / 'dd_spat_*.parquet')))
print(f'DD_SPAT files produced: {len(spat_files)}  (expected {N_IMP_CY * N_IMP_UD})')
assert len(spat_files) == N_IMP_CY * N_IMP_UD

sample = pd.read_parquet(INTERIM / 'dd_spat_1_1.parquet')
onset  = sample[sample['onset_A'] == 1]

print(f'\ndd_spat_1_1 shape: {sample.shape}')
print(f'Spatial lag columns present: {all(c in sample.columns for c in SPAT_COLS)}')
print()
print('Spatial lag distributions (onset rows):')
print(onset[SPAT_COLS].describe().round(4).to_string())

# spat_gov + spat_opp <= 1 (both are weighted fractions from the same W row)
total = onset['spat_gov'].fillna(0) + onset['spat_opp'].fillna(0)
assert (total <= 1.0 + 1e-9).all()
print('\n\u2713 spat_gov + spat_opp \u2264 1 everywhere')

# Non-onset rows must be NaN
assert sample[sample['onset_A'] != 1][SPAT_COLS].isna().all().all()
print('\u2713 Non-onset rows all NaN')

# Cold War spot-check: Afghanistan 1978
af = sample[(sample.ccode_A=='700')&(sample.ccode_B=='002')&(sample.onset_A==1)&(sample.year==1978)]
if len(af):
    assert af.iloc[0]['spat_USSR_G'] == 1
    assert af.iloc[0]['intervention'] == 2
    print('\u2713 Afghanistan 1978: USSR gov-biased, USA opp-biased')

# Self-referential zeros
usa = onset[onset['ccode_B'] == '002']
assert (usa['spat_US_G'] == 0).all() and (usa['spat_US_O'] == 0).all()
print('\u2713 USA self-referential zeros correct')

ussr = onset[onset['ccode_B'] == '364']
if len(ussr):
    assert (ussr['spat_USSR_G'] == 0).all() and (ussr['spat_USSR_O'] == 0).all()
    print('\u2713 USSR self-referential zeros correct')

print()
print('\u2713 Validation complete.')

DD_SPAT files produced: 25  (expected 25)



dd_spat_1_1 shape: (1198730, 128)
Spatial lag columns present: True

Spatial lag distributions (onset rows):
         spat_gov    spat_opp   spat_US_G  spat_USSR_G   spat_US_O  spat_USSR_O  spat_US_USRG  spat_US_USRO  spat_USR_USG  spat_USR_USO
count  25725.0000  25725.0000  25725.0000   25725.0000  25725.0000   25725.0000    25725.0000    25725.0000    25725.0000    25725.0000
mean       0.0053      0.0035      0.1188       0.0741      0.0688       0.0153        0.0006        0.0002        0.0010        0.0004
std        0.0134      0.0102      0.3236       0.2620      0.2531       0.1228        0.0241        0.0125        0.0318        0.0197
min        0.0000      0.0000      0.0000       0.0000      0.0000       0.0000        0.0000        0.0000        0.0000        0.0000
25%        0.0000      0.0000      0.0000       0.0000      0.0000       0.0000        0.0000        0.0000        0.0000        0.0000
50%        0.0000      0.0000      0.0000       0.0000      0.0000       0

✓ Non-onset rows all NaN
✓ Afghanistan 1978: USSR gov-biased, USA opp-biased
✓ USA self-referential zeros correct
✓ USSR self-referential zeros correct

✓ Validation complete.
